# PSCDL 2026 — Persistent Scene Change Detection and Localization

**Full Pipeline**: Siamese Change Detector (trained on PSCD + VL-CMU-CD) → Temporal Persistence Tracker → SAM3 Segmentation

### Papers used
| Component | Paper |
|---|---|
| Segmentation | **SAM3** (Meta, Nov 2025) — open-vocabulary detection+tracking via text/exemplar prompts. [arXiv:2511.16719](https://arxiv.org/abs/2511.16719) |
| Change Detection | **GeSCF** (CVPR 2025) — generalizable SCD via SAM pseudo-masks & geometric-semantic matching. [arXiv:2409.06214](https://arxiv.org/abs/2409.06214) |
| Backbone | **ResNet50** pretrained — Siamese feature extraction |

### Datasets
| Dataset | Pairs | Use |
|---|---|---|
| VL-CMU-CD | 3732 train + 429 test | Train change detector |
| PSCD | 770 panoramic pairs | Train change detector |
| PSCDL_2026 sample | 5 videos | Evaluate full pipeline |

### State Machine
```
T_intro  →  (wait P sec)  →  ACTIVE: SAM3 mask  →  (wait C-P sec)  →  BACKGROUND (update bg)
```
Active mask window = **[T_intro + P,  T_intro + C)**

**Kernel**: `depth-pro` (PyTorch 2.4, CUDA 12.1, SAM3 installed)

## Cell 1 — Setup & Imports

In [ ]:
import os, sys, re, json, cv2, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torch.cuda.amp import GradScaler, autocast
from PIL import Image
from pathlib import Path
from dataclasses import dataclass, field
from enum import Enum
from typing import Optional, List, Tuple, Dict
import matplotlib.pyplot as plt
import timm
warnings.filterwarnings('ignore')

os.environ['CUDA_VISIBLE_DEVICES'] = '4'   # A6000 — 49 GB VRAM
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device  : {DEVICE}  |  {torch.cuda.get_device_name(0) if DEVICE=="cuda" else "cpu"}')
print(f'PyTorch : {torch.__version__}')

SAM3_REPO = '/media/nas_mount/research3/aman_kr/midas/sam3'
if SAM3_REPO not in sys.path:
    sys.path.insert(0, SAM3_REPO)

BASE_DIR   = Path('/media/nas_mount/research3/aman_kr/vehant')
VLCMU_DIR  = BASE_DIR / 'datasets/VL-CMU-CD/VL-CMU-CD-binary255'
PSCD_DIR   = BASE_DIR / 'datasets/PSCD'
SAMPLE_DIR = BASE_DIR / 'datasets/PSCDL_2026'
CKPT_DIR   = BASE_DIR / 'models'
OUTPUT_DIR = BASE_DIR / 'outputs'
CKPT_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

SAM3_CKPT  = ('/media/nas_mount/research3/.cache/huggingface/hub/'
              'models--facebook--sam3/snapshots/'
              '3c879f39826c281e95690f02c7821c4de09afae7/sam3.pt')

print(f'VL-CMU-CD train: {len(list((VLCMU_DIR/"train/t0").iterdir()))} pairs')
print(f'VL-CMU-CD test : {len(list((VLCMU_DIR/"test/t0").iterdir()))} pairs')
print(f'PSCD           : {len(list((PSCD_DIR/"t0").iterdir()))} pairs')
print(f'Sample videos  : {len(list(SAMPLE_DIR.iterdir()))} videos')

: 

## Cell 2 — Parse Sample Video Metadata

In [ ]:
@dataclass
class VideoMeta:
    video_id: str
    video_path: Path
    duration: float
    P: float              # persistence threshold (seconds)
    C: float              # cooldown period (seconds)
    gt_intervals: List[Tuple[float, float, Path]]

def parse_metadata(video_dir: Path) -> VideoMeta:
    vid_id = video_dir.name
    txt = list(video_dir.glob('*.txt'))[0].read_text()
    duration = float(re.search(r'Video Duration:\s*(\d+)', txt).group(1))
    P = float(re.search(r'Persistence Threshold.*?:\s*(\d+)', txt).group(1))
    C = float(re.search(r'Cooldown Period.*?:\s*(\d+)', txt).group(1))
    intervals = []
    for m in re.finditer(r'(mask\d+\.png):\s*(\d+)s\s*[–\-]\s*(\d+)s', txt):
        intervals.append((float(m.group(2)), float(m.group(3)), video_dir / m.group(1)))
    mp4s = list(video_dir.glob('*.mp4'))
    return VideoMeta(vid_id, mp4s[0], duration, P, C, sorted(intervals))

video_metas = [parse_metadata(SAMPLE_DIR / f'video_{i}') for i in range(1, 6)]

print(f'{"Video":<10} {"Duration":>10} {"P":>6} {"C":>6} {"Active window":>15} {"GT intervals":>12}')
print('-'*65)
for vm in video_metas:
    print(f'{vm.video_id:<10} {vm.duration:>9.0f}s {vm.P:>5.0f}s {vm.C:>5.0f}s {vm.C-vm.P:>13.0f}s {len(vm.gt_intervals):>12}')

## Cell 3 — Dataset Loaders (PSCD + VL-CMU-CD)

In [ ]:
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import random

IMG_SIZE = 512

class ChangePairDataset(Dataset):
    """
    Generic (t0, t1) → binary change mask dataset.
    Works for both VL-CMU-CD (512×512) and PSCD (4096×1152 → random 512×512 crop).
    """
    def __init__(self, t0_dir: Path, t1_dir: Path, mask_dir: Path,
                 augment: bool = True, img_size: int = IMG_SIZE,
                 mask_threshold: int = 127):
        self.t0s    = sorted(t0_dir.glob('*.png')) + sorted(t0_dir.glob('*.jpg'))
        self.t1_dir = t1_dir
        self.mk_dir = mask_dir
        self.augment       = augment
        self.img_size      = img_size
        self.mask_threshold = mask_threshold
        self.norm = T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])

    def __len__(self): return len(self.t0s)

    def _to_tensor(self, img: Image.Image) -> torch.Tensor:
        return self.norm(torch.from_numpy(np.array(img.convert('RGB'))).permute(2,0,1).float() / 255.0)

    def __getitem__(self, idx):
        name = self.t0s[idx].name
        t0  = Image.open(self.t0s[idx]).convert('RGB')
        t1  = Image.open(self.t1_dir / name).convert('RGB')
        mk  = Image.open(self.mk_dir / name).convert('L')

        W, H = t0.size
        # Random crop for large panoramic images
        if W > self.img_size or H > self.img_size:
            x = random.randint(0, max(0, W - self.img_size))
            y = random.randint(0, max(0, H - self.img_size))
            t0 = TF.crop(t0, y, x, self.img_size, self.img_size)
            t1 = TF.crop(t1, y, x, self.img_size, self.img_size)
            mk = TF.crop(mk, y, x, self.img_size, self.img_size)
        else:
            t0 = TF.resize(t0, (self.img_size, self.img_size))
            t1 = TF.resize(t1, (self.img_size, self.img_size))
            mk = TF.resize(mk, (self.img_size, self.img_size), interpolation=TF.InterpolationMode.NEAREST)

        # Augmentation (only for training)
        if self.augment:
            if random.random() > 0.5:
                t0, t1, mk = TF.hflip(t0), TF.hflip(t1), TF.hflip(mk)
            if random.random() > 0.5:
                t0, t1, mk = TF.vflip(t0), TF.vflip(t1), TF.vflip(mk)
            if random.random() > 0.3:
                jitter = T.ColorJitter(0.3, 0.3, 0.2, 0.05)
                t0, t1 = jitter(t0), jitter(t1)

        mask = torch.from_numpy((np.array(mk) > self.mask_threshold).astype(np.float32)).unsqueeze(0)
        return self._to_tensor(t0), self._to_tensor(t1), mask


# Build combined dataset
vlcmu_train = ChangePairDataset(VLCMU_DIR/'train/t0', VLCMU_DIR/'train/t1',
                                 VLCMU_DIR/'train/mask', augment=True)
vlcmu_val   = ChangePairDataset(VLCMU_DIR/'test/t0',  VLCMU_DIR/'test/t1',
                                 VLCMU_DIR/'test/mask',  augment=False)
pscd_all    = ChangePairDataset(PSCD_DIR/'t0', PSCD_DIR/'t1',
                                 PSCD_DIR/'mask', augment=True, mask_threshold=127)

# 90/10 split for PSCD
n_pscd_val = len(pscd_all) // 10
pscd_train, pscd_val = torch.utils.data.random_split(
    pscd_all, [len(pscd_all)-n_pscd_val, n_pscd_val],
    generator=torch.Generator().manual_seed(42)
)

train_ds = ConcatDataset([vlcmu_train, pscd_train])
val_ds   = ConcatDataset([vlcmu_val,   pscd_val  ])

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False, num_workers=4, pin_memory=True)

print(f'Train: {len(train_ds):,}  |  Val: {len(val_ds):,}')
print(f'Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}')

# Quick sanity check
t0b, t1b, mkb = next(iter(train_loader))
print(f'Batch shapes — t0: {tuple(t0b.shape)}  t1: {tuple(t1b.shape)}  mask: {tuple(mkb.shape)}')

## Cell 4 — Siamese Change Detection Model

Architecture:
- **Encoder**: ResNet50 (pretrained) shared for both t0 & t1
- **Fusion**: concatenate feature difference + channel attention at 4 scales  
- **Decoder**: FPN-style upsampling → 512×512 binary mask
- **Loss**: 0.5 × BCE + 0.5 × Dice

In [ ]:
class ChannelAttention(nn.Module):
    def __init__(self, ch, r=8):
        super().__init__()
        self.fc = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                nn.Linear(ch, ch//r), nn.ReLU(),
                                nn.Linear(ch//r, ch), nn.Sigmoid())
    def forward(self, x):
        return x * self.fc(x).view(x.size(0), x.size(1), 1, 1)


class ChangeDecoder(nn.Module):
    def __init__(self, in_ch: int, out_ch: int = 64):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.block(x)


class SiameseChangeNet(nn.Module):
    """
    Siamese ResNet50 with FPN-style change decoder.
    Input : (B,3,H,W) × 2
    Output: (B,1,H,W) logits
    """
    def __init__(self, pretrained: bool = True):
        super().__init__()
        enc = timm.create_model('resnet50', pretrained=pretrained, features_only=True,
                                out_indices=(1,2,3,4))
        self.encoder = enc
        chs = enc.feature_info.channels()   # [256, 512, 1024, 2048]

        # Per-scale attention + lateral convs (on difference features)
        self.attns    = nn.ModuleList([ChannelAttention(c)   for c in chs])
        self.laterals = nn.ModuleList([nn.Conv2d(c, 128, 1)  for c in chs])

        # FPN decoder
        self.dec3 = ChangeDecoder(128, 128)
        self.dec2 = ChangeDecoder(128, 64)
        self.dec1 = ChangeDecoder(64,  32)
        self.dec0 = ChangeDecoder(32,  16)
        self.head = nn.Conv2d(16, 1, 1)

    def _diff_feats(self, t0, t1):
        f0 = self.encoder(t0)
        f1 = self.encoder(t1)
        diffs = []
        for i, (a0, a1) in enumerate(zip(f0, f1)):
            d = torch.abs(a0 - a1)
            d = self.attns[i](d)
            d = self.laterals[i](d)
            diffs.append(d)
        return diffs   # [d4, d8, d16, d32] (strides relative to input)

    def forward(self, t0, t1):
        d1, d2, d3, d4 = self._diff_feats(t0, t1)
        # d4: stride-32  d3: stride-16  d2: stride-8  d1: stride-4
        x = self.dec3(d4)
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False) + d3
        x = self.dec2(x)
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False) + d2
        x = self.dec1(x)
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False) + F.interpolate(d1, scale_factor=1, mode='bilinear', align_corners=False)
        x = self.dec0(x)
        x = F.interpolate(x, scale_factor=4, mode='bilinear', align_corners=False)
        return self.head(x)


def dice_loss(pred_logits, target, smooth=1.0):
    pred = torch.sigmoid(pred_logits)
    inter = (pred * target).sum(dim=(2,3))
    dice  = (2 * inter + smooth) / (pred.sum(dim=(2,3)) + target.sum(dim=(2,3)) + smooth)
    return 1 - dice.mean()

def change_loss(pred_logits, target):
    bce  = F.binary_cross_entropy_with_logits(pred_logits, target, pos_weight=torch.tensor(3.0, device=pred_logits.device))
    dice = dice_loss(pred_logits, target)
    return 0.5 * bce + 0.5 * dice


model = SiameseChangeNet(pretrained=True).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'SiameseChangeNet  trainable params: {n_params/1e6:.2f}M')

## Cell 5 — Training

In [ ]:
EPOCHS   = 20
LR       = 3e-4
CKPT_PATH = CKPT_DIR / 'siamese_change_best.pth'

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler    = GradScaler()


def pixel_f1(pred_logits, target, thr=0.5):
    pred = (torch.sigmoid(pred_logits) > thr).float()
    tp = (pred * target).sum()
    fp = (pred * (1-target)).sum()
    fn = ((1-pred) * target).sum()
    p  = tp / (tp+fp+1e-8)
    r  = tp / (tp+fn+1e-8)
    return (2*p*r/(p+r+1e-8)).item()


train_losses, val_f1s = [], []
best_f1 = 0.0

for epoch in range(1, EPOCHS+1):
    # ── Train ──────────────────────────────────────────────────────────────────
    model.train()
    ep_loss = 0.0
    for t0b, t1b, mkb in train_loader:
        t0b, t1b, mkb = t0b.to(DEVICE), t1b.to(DEVICE), mkb.to(DEVICE)
        optimizer.zero_grad()
        with autocast():
            logits = model(t0b, t1b)
            loss   = change_loss(logits, mkb)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        ep_loss += loss.item()
    ep_loss /= len(train_loader)

    # ── Validate ───────────────────────────────────────────────────────────────
    model.eval()
    f1_scores = []
    with torch.no_grad():
        for t0b, t1b, mkb in val_loader:
            t0b, t1b, mkb = t0b.to(DEVICE), t1b.to(DEVICE), mkb.to(DEVICE)
            with autocast():
                logits = model(t0b, t1b)
            f1_scores.append(pixel_f1(logits, mkb))
    val_f1 = float(np.mean(f1_scores))

    scheduler.step()
    train_losses.append(ep_loss)
    val_f1s.append(val_f1)

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), CKPT_PATH)
        marker = '  ← best'
    else:
        marker = ''

    if epoch % 2 == 0 or epoch == 1:
        print(f'Epoch {epoch:2d}/{EPOCHS}  loss={ep_loss:.4f}  val_F1={val_f1:.4f}{marker}')

print(f'\nBest val F1: {best_f1:.4f}  (saved to {CKPT_PATH})')

## Cell 6 — Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(train_losses, marker='o', ms=4)
ax1.set_title('Training Loss (BCE + Dice)')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.grid(alpha=0.3)

ax2.plot(val_f1s, marker='o', ms=4, color='green')
ax2.axhline(best_f1, color='red', ls='--', label=f'Best={best_f1:.4f}')
ax2.set_title('Validation F1 (PSCD + VL-CMU-CD)')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('F1'); ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=100)
plt.show()

## Cell 7 — Qualitative Check: Change Detection on Image Pairs

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
model.eval()

norm = T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])

@torch.inference_mode()
def predict_change(t0_path: str, t1_path: str, thr: float = 0.5) -> np.ndarray:
    def load(p):
        img = Image.open(p).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
        return norm(torch.from_numpy(np.array(img)).permute(2,0,1).float()/255).unsqueeze(0).to(DEVICE)
    logits = model(load(t0_path), load(t1_path))
    return (torch.sigmoid(logits).squeeze().cpu().numpy() > thr).astype(np.uint8)


# Show 4 examples from VL-CMU-CD test set
test_imgs = sorted((VLCMU_DIR/'test/t0').iterdir())[:4]
fig, axes = plt.subplots(4, 4, figsize=(14, 14))

for row, img_path in enumerate(test_imgs):
    name = img_path.name
    t0   = Image.open(VLCMU_DIR/'test/t0'/name).convert('RGB')
    t1   = Image.open(VLCMU_DIR/'test/t1'/name).convert('RGB')
    gt   = np.array(Image.open(VLCMU_DIR/'test/mask'/name).convert('L'))
    gt   = (gt > 127).astype(np.uint8)
    pred = predict_change(VLCMU_DIR/'test/t0'/name, VLCMU_DIR/'test/t1'/name)

    tp = np.logical_and(pred>0, gt>0)
    fp = np.logical_and(pred>0, gt==0)
    fn = np.logical_and(pred==0, gt>0)
    overlay = np.zeros((*gt.shape, 3), dtype=np.uint8)
    overlay[tp] = [0,0,255]; overlay[fp] = [255,0,0]; overlay[fn] = [0,255,0]

    p = tp.sum()/(tp.sum()+fp.sum()+1e-8)
    r = tp.sum()/(tp.sum()+fn.sum()+1e-8)
    f = 2*p*r/(p+r+1e-8)

    axes[row][0].imshow(t0.resize((256,256)))
    axes[row][0].set_title('Before (t0)'); axes[row][0].axis('off')
    axes[row][1].imshow(t1.resize((256,256)))
    axes[row][1].set_title('After (t1)'); axes[row][1].axis('off')
    axes[row][2].imshow(gt, cmap='gray')
    axes[row][2].set_title('GT mask'); axes[row][2].axis('off')
    axes[row][3].imshow(overlay)
    axes[row][3].set_title(f'Pred  F1={f:.3f}\nBlue=TP Red=FP Green=FN'); axes[row][3].axis('off')

plt.suptitle('SiameseChangeNet — VL-CMU-CD test examples', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'change_det_examples.png', dpi=100)
plt.show()

## Cell 8 — Load SAM3

In [ ]:
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

print('Loading SAM3...')
sam3_model = build_sam3_image_model(checkpoint_path=SAM3_CKPT).to(DEVICE).eval()
sam3_proc  = Sam3Processor(sam3_model, resolution=1008, device=DEVICE)
print('SAM3 loaded ✓')


@torch.inference_mode()
def sam3_refine(frame_bgr: np.ndarray, coarse_mask: np.ndarray,
                text_prompt: str = 'abandoned object encroachment') -> np.ndarray:
    """
    Refine a coarse change mask using SAM3 (text + box prompt).
    coarse_mask: (H,W) binary uint8
    Returns refined (H,W) binary uint8 mask.
    """
    H, W = frame_bgr.shape[:2]
    # Find bounding box of coarse mask
    ys, xs = np.where(coarse_mask > 0)
    if len(ys) == 0:
        return coarse_mask
    x1, y1, x2, y2 = int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())
    # Expand bbox by 10%
    pad_x, pad_y = max(5, (x2-x1)//10), max(5, (y2-y1)//10)
    x1 = max(0, x1-pad_x); y1 = max(0, y1-pad_y)
    x2 = min(W, x2+pad_x); y2 = min(H, y2+pad_y)

    pil   = Image.fromarray(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
    state = sam3_proc.set_image(pil)
    state = sam3_proc.set_text_prompt(text_prompt, state)

    cx = ((x1+x2)/2)/W; cy = ((y1+y2)/2)/H
    bw = (x2-x1)/W;     bh = (y2-y1)/H
    sam3_proc.add_geometric_prompt([cx, cy, bw, bh], label=True, state=state)

    masks  = state.get('masks')
    scores = state.get('scores')

    if masks is None or (hasattr(masks,'__len__') and len(masks)==0):
        return coarse_mask  # fallback to coarse

    best = int(torch.argmax(scores) if torch.is_tensor(scores) else np.argmax(np.asarray(scores)))
    m = masks[best]
    if torch.is_tensor(m):
        while m.dim() > 2: m = m.squeeze(0)
        m = (m > 0.5).cpu().numpy().astype(np.uint8)
    else:
        m = (np.asarray(m) > 0.5).astype(np.uint8)

    if m.shape != (H, W):
        m = cv2.resize(m, (W, H), interpolation=cv2.INTER_NEAREST)
    return m

## Cell 9 — Background Model + Persistence Tracker

In [ ]:
IMG_NORM = T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])

def build_background(video_path: Path, n_frames: int = 300, every: int = 5) -> np.ndarray:
    cap = cv2.VideoCapture(str(video_path))
    frames, i = [], 0
    while cap.isOpened() and len(frames) < n_frames:
        ret, f = cap.read()
        if not ret: break
        if i % every == 0: frames.append(f)
        i += 1
    cap.release()
    return np.median(np.stack(frames), axis=0).astype(np.uint8)


@torch.inference_mode()
def siamese_change_mask(bg: np.ndarray, frame: np.ndarray, thr: float = 0.5) -> np.ndarray:
    """Run SiameseChangeNet on (bg, current_frame) → binary change mask."""
    def prep(img_bgr):
        rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        t = torch.from_numpy(cv2.resize(rgb,(IMG_SIZE,IMG_SIZE))).permute(2,0,1).float()/255
        return IMG_NORM(t).unsqueeze(0).to(DEVICE)
    logits = model(prep(bg), prep(frame))
    prob   = torch.sigmoid(logits).squeeze().cpu().numpy()
    mask   = (prob > thr).astype(np.uint8)
    H, W   = frame.shape[:2]
    return cv2.resize(mask, (W, H), interpolation=cv2.INTER_NEAREST)


# ── State machine ─────────────────────────────────────────────────────────────
class ObjState(Enum):
    DETECTED   = 'detected'
    ACTIVE     = 'active'
    BACKGROUND = 'background'

@dataclass
class TrackedObj:
    obj_id:  int
    t_intro: float
    bbox:    Tuple
    mask:    Optional[np.ndarray] = None
    state:   ObjState = ObjState.DETECTED

    def tick(self, t, P, C):
        e = t - self.t_intro
        if e >= C:   self.state = ObjState.BACKGROUND
        elif e >= P: self.state = ObjState.ACTIVE
        return self.state

def _iou(b1, b2):
    xi1,yi1=max(b1[0],b2[0]),max(b1[1],b2[1])
    xi2,yi2=min(b1[2],b2[2]),min(b1[3],b2[3])
    inter=max(0,xi2-xi1)*max(0,yi2-yi1)
    if inter==0: return 0.0
    return inter/((b1[2]-b1[0])*(b1[3]-b1[1])+(b2[2]-b2[0])*(b2[3]-b2[1])-inter)

class ObjectTracker:
    def __init__(self, P, C, fps, min_area=800, iou_thr=0.2, pre_frames=25):
        self.P, self.C, self.fps = P, C, fps
        self.min_area, self.iou_thr = min_area, iou_thr
        self.pre_frames  = pre_frames   # frames blob must persist before registering
        self._objs: Dict[int,TrackedObj] = {}
        self._cands: Dict[tuple,int]     = {}
        self._next_id = 0
        self._bg_ids: set = set()

    def step(self, t, blobs, frame_bgr, coarse_full):
        unmatched = list(range(len(blobs)))
        for obj in [o for o in self._objs.values() if o.state != ObjState.BACKGROUND]:
            bi, bv = -1, 0.0
            for i in unmatched:
                v = _iou(obj.bbox, blobs[i][0])
                if v > bv: bv, bi = v, i
            if bv >= self.iou_thr and bi >= 0:
                obj.bbox = blobs[bi][0]; unmatched.remove(bi)

        for i in unmatched:
            bbox, area = blobs[i]
            if area < self.min_area: continue
            key = (bbox[0]//25, bbox[1]//25, bbox[2]//25, bbox[3]//25)
            self._cands[key] = self._cands.get(key, 0) + 1
            if self._cands[key] >= self.pre_frames:
                self._objs[self._next_id] = TrackedObj(self._next_id, t, bbox)
                self._next_id += 1
                del self._cands[key]

        newly_bg = []
        for obj in self._objs.values():
            prev = obj.state
            obj.tick(t, self.P, self.C)
            if prev == ObjState.DETECTED and obj.state == ObjState.ACTIVE:
                # Extract coarse mask for this bbox, then refine with SAM3
                x1,y1,x2,y2 = obj.bbox
                local_coarse = np.zeros_like(coarse_full)
                local_coarse[y1:y2, x1:x2] = coarse_full[y1:y2, x1:x2]
                obj.mask = sam3_refine(frame_bgr, local_coarse)
            if obj.state == ObjState.BACKGROUND and obj.obj_id not in self._bg_ids:
                self._bg_ids.add(obj.obj_id); newly_bg.append(obj)

        active = [o for o in self._objs.values() if o.state == ObjState.ACTIVE]
        return active, newly_bg

    def absorb_background(self, bg, frame, newly_bg, alpha=0.3):
        for obj in newly_bg:
            if obj.mask is not None:
                m = obj.mask[:,:,None]
                bg = (bg*(1-alpha*m) + frame*alpha*m).astype(np.uint8)
        return bg

## Cell 10 — Metrics

In [ ]:
def compute_f1(pred: np.ndarray, gt: np.ndarray) -> Dict:
    pred, gt = (pred>0).astype(bool), (gt>0).astype(bool)
    tp = np.logical_and( pred,  gt).sum()
    fp = np.logical_and( pred, ~gt).sum()
    fn = np.logical_and(~pred,  gt).sum()
    p = tp/(tp+fp+1e-8); r = tp/(tp+fn+1e-8)
    return dict(f1=float(2*p*r/(p+r+1e-8)), precision=float(p), recall=float(r))

def load_gt(t: float, meta: VideoMeta, H: int, W: int) -> np.ndarray:
    for t0,t1,mp in meta.gt_intervals:
        if t0 <= t <= t1:
            g = np.array(Image.open(mp).convert('L'))
            return (cv2.resize(g,(W,H),interpolation=cv2.INTER_NEAREST) > 127).astype(np.uint8)
    return np.zeros((H,W), dtype=np.uint8)

## Cell 11 — Full Pipeline (per video)

In [ ]:
def run_pipeline(meta: VideoMeta, eval_every_sec=1.0,
                 change_thr=0.45, min_area=800, bg_frames=300) -> Dict:
    print(f'\n{"="*60}')
    print(f'{meta.video_id}  |  P={meta.P}s  C={meta.C}s  dur={meta.duration}s')

    bg = build_background(meta.video_path, n_frames=bg_frames)
    H, W = bg.shape[:2]

    mog2 = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=50, detectShadows=True)
    for _ in range(60): mog2.apply(bg)

    cap = cv2.VideoCapture(str(meta.video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    tracker = ObjectTracker(meta.P, meta.C, fps, min_area=min_area,
                            pre_frames=max(5, int(fps*2.5)))

    results = []
    current_mask = np.zeros((H,W), dtype=np.uint8)
    eval_step    = max(1, int(fps * eval_every_sec))
    frame_idx    = 0
    SUBSAMPLE    = 3   # process every 3rd frame for speed (still ~8fps)

    print(f'  Processing at {fps/SUBSAMPLE:.1f} effective fps...')
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break

        t = frame_idx / fps

        if frame_idx % SUBSAMPLE == 0:
            # ── Siamese change map ────────────────────────────────────────────
            coarse = siamese_change_mask(bg, frame, thr=change_thr)

            # ── MOG2 gate (removes illumination artefacts) ────────────────────
            mog = mog2.apply(frame)
            mog = cv2.threshold(mog, 200, 255, cv2.THRESH_BINARY)[1]
            kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))
            mog = cv2.morphologyEx(mog, cv2.MORPH_OPEN, kernel, iterations=1)

            # Combine: siamese AND (mog OR close-to-prev-mask)
            combined = cv2.bitwise_and(coarse, (mog > 0).astype(np.uint8))
            combined = cv2.morphologyEx(combined, cv2.MORPH_CLOSE, kernel, iterations=2)

            # Blobs
            n, _, stats, _ = cv2.connectedComponentsWithStats(combined)
            blobs = []
            for lbl in range(1, n):
                area = stats[lbl, cv2.CC_STAT_AREA]
                if area < min_area: continue
                x1=stats[lbl,cv2.CC_STAT_LEFT]; y1=stats[lbl,cv2.CC_STAT_TOP]
                x2=x1+stats[lbl,cv2.CC_STAT_WIDTH]; y2=y1+stats[lbl,cv2.CC_STAT_HEIGHT]
                blobs.append(((x1,y1,x2,y2), area))

            # ── Tracker step ─────────────────────────────────────────────────
            active, newly_bg = tracker.step(t, blobs, frame, coarse)

            if newly_bg:
                bg = tracker.absorb_background(bg, frame, newly_bg)
                print(f'    t={t:.1f}s  absorbed {len(newly_bg)} obj(s) into background')

            if active:
                current_mask = np.zeros((H,W), dtype=np.uint8)
                for obj in active:
                    if obj.mask is not None:
                        current_mask = np.logical_or(current_mask, obj.mask).astype(np.uint8)
                    else:
                        x1,y1,x2,y2 = obj.bbox
                        current_mask[y1:y2,x1:x2] = 1
            else:
                current_mask = np.zeros((H,W), dtype=np.uint8)

        if frame_idx % eval_step == 0:
            gt = load_gt(t, meta, H, W)
            m  = compute_f1(current_mask, gt)
            results.append(dict(t=t, metrics=m, pred=current_mask.copy(), gt=gt))

        frame_idx += 1
    cap.release()

    f1s = [r['metrics']['f1']        for r in results]
    prs = [r['metrics']['precision'] for r in results]
    rcs = [r['metrics']['recall']    for r in results]
    summary = dict(video_id=meta.video_id,
                   mean_f1=float(np.mean(f1s)),
                   mean_prec=float(np.mean(prs)),
                   mean_rec=float(np.mean(rcs)),
                   intervals=results)
    print(f'  ✓ F1={summary["mean_f1"]:.4f}  Prec={summary["mean_prec"]:.4f}  Rec={summary["mean_rec"]:.4f}')
    return summary

## Cell 12 — Run on All 5 Sample Videos

In [ ]:
all_results = []
for vm in video_metas:
    r = run_pipeline(vm)
    all_results.append(r)

print('\n' + '='*60)
print(f'{"Video":<12} {"F1":>8} {"Precision":>12} {"Recall":>10}')
print('-'*60)
for r in all_results:
    print(f'{r["video_id"]:<12} {r["mean_f1"]:>8.4f} {r["mean_prec"]:>12.4f} {r["mean_rec"]:>10.4f}')
print('-'*60)
mf1 = np.mean([r['mean_f1'] for r in all_results])
print(f'{"OVERALL":<12} {mf1:>8.4f}')

## Cell 13 — F1 Over Time & Mask Visualizations

In [ ]:
fig, axes = plt.subplots(len(all_results), 1, figsize=(14, 3*len(all_results)))
for ax, result, meta in zip(axes, all_results, video_metas):
    ts  = [r['t']           for r in result['intervals']]
    f1s = [r['metrics']['f1'] for r in result['intervals']]
    ax.plot(ts, f1s, lw=1.5, color='steelblue', label='F1')
    ax.axhline(result['mean_f1'], color='red', ls='--', lw=1, label=f'Mean={result["mean_f1"]:.3f}')
    for t0,t1,mp in meta.gt_intervals:
        if np.array(Image.open(mp).convert('L')).max() > 10:
            ax.axvspan(t0, t1, alpha=0.15, color='green')
    ax.set(title=f'{meta.video_id}  P={meta.P}s C={meta.C}s',
           xlabel='Time (s)', ylabel='F1', ylim=(0,1.05))
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.suptitle('F1 over time  (green shading = GT active change)', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'f1_over_time.png', dpi=100)
plt.show()

# Mask comparison for first video
result = all_results[0]; meta = video_metas[0]
samples = [result['intervals'][i] for i in np.linspace(0, len(result['intervals'])-1, 5, dtype=int)]
fig, axes = plt.subplots(5, 3, figsize=(12, 15))
for ax_row, s in zip(axes, samples):
    pred, gt = s['pred'], s['gt']
    vis = np.zeros((*gt.shape,3), dtype=np.uint8)
    vis[np.logical_and(pred>0,gt>0)] = [0,0,255]
    vis[np.logical_and(pred>0,gt==0)] = [255,0,0]
    vis[np.logical_and(pred==0,gt>0)] = [0,255,0]
    ax_row[0].imshow(pred, cmap='gray'); ax_row[0].set_title(f't={s["t"]:.0f}s  Predicted'); ax_row[0].axis('off')
    ax_row[1].imshow(gt,   cmap='gray'); ax_row[1].set_title('Ground Truth');                ax_row[1].axis('off')
    ax_row[2].imshow(vis);               ax_row[2].set_title(f'F1={s["metrics"]["f1"]:.3f}  B=TP R=FP G=FN'); ax_row[2].axis('off')
plt.suptitle(f'{meta.video_id} — Mask Comparison', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/f'{meta.video_id}_masks.png', dpi=100)
plt.show()

## Cell 14 — Generate Submission Masks (Phase 2 Test Videos)

In [ ]:
def generate_submission(video_path: Path, P: float, C: float,
                         out_dir: Path, min_area=800):
    """
    Run pipeline on a test video (no ground truth).
    Outputs maskN.png + intervals.txt — one mask per state transition.
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    bg = build_background(video_path)
    H, W = bg.shape[:2]

    mog2 = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=50, detectShadows=True)
    for _ in range(60): mog2.apply(bg)

    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    tracker = ObjectTracker(P, C, fps, min_area=min_area,
                            pre_frames=max(5, int(fps*2.5)))

    intervals = []
    prev_mask = np.zeros((H,W), dtype=np.uint8)
    seg_start = 0.0
    frame_idx = 0
    SUBSAMPLE = 3
    current_mask = np.zeros((H,W), dtype=np.uint8)

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        t = frame_idx / fps

        if frame_idx % SUBSAMPLE == 0:
            coarse = siamese_change_mask(bg, frame, thr=0.45)
            mog = cv2.threshold(mog2.apply(frame), 200, 255, cv2.THRESH_BINARY)[1]
            mog = cv2.morphologyEx(mog, cv2.MORPH_OPEN,
                                    cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(5,5)))
            combined = cv2.morphologyEx(
                cv2.bitwise_and(coarse,(mog>0).astype(np.uint8)),
                cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(5,5)), iterations=2)
            n,_,stats,_ = cv2.connectedComponentsWithStats(combined)
            blobs = []
            for lbl in range(1,n):
                a=stats[lbl,cv2.CC_STAT_AREA]
                if a<min_area: continue
                x1=stats[lbl,cv2.CC_STAT_LEFT]; y1=stats[lbl,cv2.CC_STAT_TOP]
                blobs.append(((x1,y1,x1+stats[lbl,cv2.CC_STAT_WIDTH],y1+stats[lbl,cv2.CC_STAT_HEIGHT]),a))
            active, newly_bg = tracker.step(t, blobs, frame, coarse)
            if newly_bg: bg = tracker.absorb_background(bg, frame, newly_bg)
            current_mask = np.zeros((H,W), dtype=np.uint8)
            for obj in active:
                if obj.mask is not None:
                    current_mask = np.logical_or(current_mask, obj.mask).astype(np.uint8)

        if not np.array_equal(current_mask, prev_mask):
            intervals.append((seg_start, t, prev_mask.copy()))
            seg_start = t; prev_mask = current_mask.copy()
        frame_idx += 1

    intervals.append((seg_start, frame_idx/fps, prev_mask.copy()))
    cap.release()

    with open(out_dir/'intervals.txt','w') as f:
        for i,(t0,t1,mask) in enumerate(intervals,1):
            cv2.imwrite(str(out_dir/f'mask{i}.png'), mask*255)
            f.write(f'mask{i}.png: {t0:.0f}s – {t1:.0f}s\n')
    print(f'Saved {len(intervals)} masks to {out_dir}')


# Run on all sample videos as a dry-run
for vm in video_metas:
    generate_submission(vm.video_path, vm.P, vm.C,
                         OUTPUT_DIR/f'submission_{vm.video_id}')

## Cell 15 — Final Results Summary

In [ ]:
print('='*60)
print('PSCDL 2026 — Final Results')
print('Method: SiameseResNet50 + MOG2 + State Machine + SAM3')
print('Trained on: VL-CMU-CD (3732+429) + PSCD (693+77)')
print('='*60)
print(f'{"Video":<12} {"F1":>8} {"Prec":>8} {"Rec":>8}')
print('-'*60)
for r in all_results:
    print(f'{r["video_id"]:<12} {r["mean_f1"]:>8.4f} {r["mean_prec"]:>8.4f} {r["mean_rec"]:>8.4f}')
print('-'*60)
mf1 = np.mean([r['mean_f1']   for r in all_results])
mpr = np.mean([r['mean_prec'] for r in all_results])
mrc = np.mean([r['mean_rec']  for r in all_results])
print(f'{"OVERALL":<12} {mf1:>8.4f} {mpr:>8.4f} {mrc:>8.4f}')
print('='*60)

summary = dict(
    method='SiameseResNet50+MOG2+StateMachine+SAM3',
    trained_on='VL-CMU-CD+PSCD',
    overall=dict(f1=mf1, precision=mpr, recall=mrc),
    per_video=[dict(id=r['video_id'], f1=r['mean_f1'],
                    precision=r['mean_prec'], recall=r['mean_rec'])
               for r in all_results]
)
with open(OUTPUT_DIR/'results_final.json','w') as f:
    json.dump(summary, f, indent=2)
print(f'Results saved → {OUTPUT_DIR}/results_final.json')